In [1]:
import re
import requests

from pathlib import Path

import pandas as pd

from bs4 import BeautifulSoup

In [2]:
URLS = {
    "2019-2022": "https://www.ncsl.org/technology-and-communication/legislation-related-to-artificial-intelligence",
    "2023": "https://www.ncsl.org/technology-and-communication/artificial-intelligence-2023-legislation",
    "2024": "https://www.ncsl.org/technology-and-communication/artificial-intelligence-2024-legislation",
    "2025": "https://www.ncsl.org/technology-and-communication/artificial-intelligence-2025-legislation",
}

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; research-scraper/1.0)"}

RAW_DIR = Path("../../data/2018-2025/raw_html")

# NCSL names the state column differently on some pages
COLUMN_RENAMES = {"Jurisdiction": "State", "Jurisdiction and Summary": "State"}

# The "2019-2022" page has tables listed in reverse-chronological order
EXPECTED_YEARS = {"2019-2022": [2022, 2021, 2020, 2019],
                  "2023": [2023], "2024": [2024], "2025": [2025]}

In [3]:
def clean(s):
    return re.sub(r"\s+", " ", s).strip()


def get_tables(page, url, expected_years):
    """Scrape every table on one NCSL page into a list of DataFrames"""
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()

    RAW_DIR.mkdir(parents=True, exist_ok=True)
    raw_file_name = RAW_DIR / f"ncsl_ai_db_{page}.html"
    raw_file_name.write_text(resp.text, encoding="utf-8")

    soup = BeautifulSoup(resp.text, "lxml")
    tables = [t for t in soup.find_all("table") if t.find("tr")]
    assert len(tables) == len(expected_years), \
        f"{page}: expected {len(expected_years)} tables, found {len(tables)} - page layout may have changed"

    frames = []
    for table, year in zip(tables, expected_years):
        rows = table.find_all("tr")

        # First row of table data contains headers
        headers = [clean(c.get_text(" "))
                   for c in rows[0].find_all(["th", "td"])]
        n_cols = len(headers)

        table_data = []
        for row in rows[1:]:
            data = [clean(c.get_text(" ")) for c in row.find_all(["td", "th"])]

            # First link in the row that has an href ("" if there is none)
            row_url = next((a["href"]
                           for a in row.find_all("a", href=True)), "")

            if not (any(data) or row_url):
                continue

            # Pad rows with no bill information, then add URL
            data = (data + [""] * n_cols)[:n_cols]
            data.append(row_url)
            table_data.append(data)

        df = pd.DataFrame(table_data, columns=headers +
                          ["URL"]).rename(columns=COLUMN_RENAMES)
        df["Page Year"] = year
        frames.append(df)

    return frames

In [4]:
# Scrape each web page into a dataframe by year
legi_data = []
for page, url in URLS.items():
    legi_data.extend(get_tables(page, url, EXPECTED_YEARS[page]))

# Make sure each year is accounted for within dataset
assert len(legi_data) == 7

In [5]:
# Combine and export data to a csv
legi_data_df = pd.concat(legi_data, ignore_index=True)

file_name = "../../data/2018-2025/ncsl_ai_db_2018_2025.csv"
legi_data_df.to_csv(file_name, index=False)